In [91]:
# ============================================================
# STEP 2A — Load and Verify SHAP Sample
# ============================================================

# Load the 600-instance stratified sample
shap_sample_df = pd.read_csv("stratified_samples.csv")

print("SHAP Sample Verification")
print("=" * 60)

# Basic shape
print(f"Sample shape: {shap_sample_df.shape}")

# Required columns
print(f"\nColumns:")
print(shap_sample_df.columns.tolist())

# Stratum counts
print("\nStratum counts:")
print(
    shap_sample_df["uncertainty_stratum"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"])
)

# Check unique instance IDs
print(f"\nUnique test instance IDs: "
      f"{shap_sample_df['test_instance_id'].nunique()}")

# Check ID range
print(f"Minimum test instance ID: "
      f"{shap_sample_df['test_instance_id'].min()}")

print(f"Maximum test instance ID: "
      f"{shap_sample_df['test_instance_id'].max()}")

# Check whether every ID is valid for the test set
valid_ids = (
    (shap_sample_df["test_instance_id"] >= 0) &
    (shap_sample_df["test_instance_id"] < len(X_test_processed))
)

print(f"\nAll instance IDs valid: {valid_ids.all()}")

# Check expected sample size
print(f"Exactly 600 instances: "
      f"{len(shap_sample_df) == 600}")

# Check exactly 200 per stratum
stratum_counts = (
    shap_sample_df["uncertainty_stratum"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH"], fill_value=0)
)

print(
    f"Exactly 200 per stratum: "
    f"{all(stratum_counts == 200)}"
)

# ------------------------------------------------------------
# Retrieve the corresponding feature rows
# ------------------------------------------------------------

shap_instance_ids = shap_sample_df["test_instance_id"].to_numpy()

X_shap = X_test_processed.iloc[shap_instance_ids].copy()
y_shap = y_test_processed[shap_instance_ids]
print("\nSHAP Input Data")
print("=" * 60)
print(f"X_shap shape: {X_shap.shape}")
print(f"y_shap shape: {y_shap.shape}")

print("\nFirst 5 instance IDs:")
print(shap_instance_ids[:5])

print("\nFirst 5 true labels:")
print(y_shap[:5])

print("\nFeature count:")
print(f"{X_shap.shape[1]} features")

SHAP Sample Verification
Sample shape: (600, 3)

Columns:
['test_instance_id', 'sigma_squared', 'uncertainty_stratum']

Stratum counts:
uncertainty_stratum
LOW       200
MEDIUM    200
HIGH      200
Name: count, dtype: int64

Unique test instance IDs: 600
Minimum test instance ID: 481
Maximum test instance ID: 37842

All instance IDs valid: True
Exactly 600 instances: True
Exactly 200 per stratum: True

SHAP Input Data
X_shap shape: (600, 21)
y_shap shape: (600,)

First 5 instance IDs:
[ 529  850  905 1029 1041]

First 5 true labels:
[0. 0. 0. 1. 1.]

Feature count:
21 features


In [92]:
# Retrieve labels for the 600 SHAP instances
y_shap = y_test_processed[shap_instance_ids]

print("\nSHAP Input Data")
print("=" * 60)
print(f"X_shap shape: {X_shap.shape}")
print(f"y_shap shape: {y_shap.shape}")

print("\nFirst 5 instance IDs:")
print(shap_instance_ids[:5])

print("\nFirst 5 true labels:")
print(y_shap[:5])

print("\nFeature count:")
print(f"{X_shap.shape[1]} features")


SHAP Input Data
X_shap shape: (600, 21)
y_shap shape: (600,)

First 5 instance IDs:
[ 529  850  905 1029 1041]

First 5 true labels:
[0. 0. 0. 1. 1.]

Feature count:
21 features


In [93]:
# ============================================================
# STEP 2B — SHAP Explainer Setup
# ============================================================

import shap
import numpy as np
import torch

# ------------------------------------------------------------
# SHAP prediction function
# ------------------------------------------------------------

def shap_predict(data):
    """
    SHAP-compatible prediction function.

    Input:
        data -> NumPy array of shape (n_instances, 21)

    Output:
        Diabetes probability for each instance.
    """

    tensor_data = torch.tensor(
        data,
        dtype=torch.float32
    )

    model.eval()

    with torch.no_grad():
        predictions = model(tensor_data).squeeze(1).cpu().numpy()

    return predictions


# ------------------------------------------------------------
# Create SHAP background dataset
# ------------------------------------------------------------

SHAP_BACKGROUND_SIZE = 500

np.random.seed(42)

background_indices = np.random.choice(
    len(X_train_processed),
    size=SHAP_BACKGROUND_SIZE,
    replace=False
)

X_shap_background = X_train_processed.iloc[
    background_indices
].to_numpy(dtype=np.float32)

print("SHAP Background Dataset")
print("=" * 60)
print(f"Background shape: {X_shap_background.shape}")

# ------------------------------------------------------------
# Create SHAP explainer
# ------------------------------------------------------------

shap_explainer = shap.Explainer(
    shap_predict,
    X_shap_background,
    feature_names=X_shap.columns.tolist()
)

print("\nSHAP Explainer")
print("=" * 60)
print(f"Explainer type: {type(shap_explainer).__name__}")
print("Explainer created successfully.")

SHAP Background Dataset
Background shape: (500, 21)

SHAP Explainer
Explainer type: PermutationExplainer
Explainer created successfully.


In [94]:
# ============================================================
# STEP 2C — SHAP Single-Instance Verification
# ============================================================

# Explain the first sampled test instance
X_single_shap = X_shap.iloc[[0]].to_numpy(dtype=np.float32)

# Generate SHAP explanation
shap_single = shap_explainer(
    X_single_shap
)

print("Single-Instance SHAP Verification")
print("=" * 60)

print(f"Input shape:       {X_single_shap.shape}")
print(f"SHAP values shape:  {shap_single.values.shape}")

print(f"\nInstance ID:       {shap_instance_ids[0]}")
print(f"True label:        {y_shap[0]:.0f}")

# Model prediction
single_prediction = shap_predict(X_single_shap)[0]

print(f"Model probability: {single_prediction:.6f}")

# Expected value / base value
print(f"Base value:        {shap_single.base_values[0]}")

# Feature attribution check
print("\nFeature attributions:")
for feature, value in zip(
    X_shap.columns,
    shap_single.values[0]
):
    print(f"{feature:<25} {value:+.8f}")

# ------------------------------------------------------------
# Additivity check
# ------------------------------------------------------------

base_value = np.asarray(shap_single.base_values[0]).item()
shap_sum = shap_single.values[0].sum()

print("\nAdditivity Check")
print("=" * 60)
print(f"Base value + SHAP sum: {base_value + shap_sum:.6f}")
print(f"Model probability:     {single_prediction:.6f}")
print(
    f"Absolute difference:   "
    f"{abs((base_value + shap_sum) - single_prediction):.10f}"
)

PermutationExplainer explainer: 2it [00:10, 10.39s/it]               

Single-Instance SHAP Verification
Input shape:       (1, 21)
SHAP values shape:  (1, 21)

Instance ID:       529
True label:        0
Model probability: 0.501891
Base value:        0.31813145988620817

Feature attributions:
HighBP                    +0.06000653
HighChol                  -0.04258002
CholCheck                 +0.00160859
BMI                       -0.10004776
Smoker                    +0.00501761
Stroke                    -0.00049158
HeartDiseaseorAttack      -0.00359675
PhysActivity              -0.00182458
Fruits                    +0.00323367
Veggies                   +0.00705422
HvyAlcoholConsump         +0.00311083
AnyHealthcare             +0.00100250
NoDocbcCost               +0.00056265
GenHlth                   +0.19585650
MentHlth                  +0.00523255
PhysHlth                  -0.02010542
DiffWalk                  -0.00564787
Sex                       +0.01365392
Age                       +0.03682720
Education                 +0.00081586
Income          

In [95]:
# ============================================================
# STEP 2D — SHAP on All 600 Sampled Instances
# ============================================================

print("Starting SHAP computation for 600 instances...")
print("=" * 60)

# Convert the 600 sampled instances to NumPy
X_shap_array = X_shap.to_numpy(dtype=np.float32)

# Compute SHAP explanations
shap_explanation = shap_explainer(
    X_shap_array
)

# Extract SHAP values
shap_values = shap_explanation.values

print("\nSHAP Computation Completed")
print("=" * 60)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected shape:    (600, 21)")

# ------------------------------------------------------------
# Verify SHAP output
# ------------------------------------------------------------

print("\nVerification")
print("=" * 60)

print(f"Number of instances: {shap_values.shape[0]}")
print(f"Number of features:  {shap_values.shape[1]}")

print(
    f"Contains NaN: "
    f"{np.isnan(shap_values).any()}"
)

print(
    f"Contains Inf: "
    f"{np.isinf(shap_values).any()}"
)

# ------------------------------------------------------------
# Save SHAP values
# ------------------------------------------------------------

np.save(
    "shap_values.npy",
    shap_values
)

# Save corresponding instance IDs
np.save(
    "shap_instance_ids.npy",
    shap_instance_ids
)

print("\nFiles Saved")
print("=" * 60)
print("shap_values.npy")
print("shap_instance_ids.npy")

Starting SHAP computation for 600 instances...


PermutationExplainer explainer: 601it [00:11,  7.22it/s]                         


SHAP Computation Completed
SHAP values shape: (600, 21)
Expected shape:    (600, 21)

Verification
Number of instances: 600
Number of features:  21
Contains NaN: False
Contains Inf: False

Files Saved
shap_values.npy
shap_instance_ids.npy
